# 第 10 天：动量因子 1

> 来自《30 天因子研究计划》第 10 天  
> 主题：动量因子 1  
> 必做：20/60 日动量  
> 选做：120 日动量  
> 目标产出：动量因子库

---

## 0. 今天你要真正学会什么？

前 6-9 天我们学习了基本面因子：价值和质量。  
今天开始进入价格行为因子：动量。

动量因子的核心问题是：


过去表现强的股票，未来是否仍然更强？


今天重点掌握：

1. 20 日、60 日、120 日动量如何计算。
2. 为什么动量因子要注意收益窗口和跳过短期反转。
3. 如何把价格宽表变成动量因子库。
4. 如何用 IC 和分组收益快速检查动量有效性。

一句话版：

> 动量因子不是追涨杀跌的口号，而是把“趋势延续”变成可计算、可检验的历史收益信号。

---

## 1. 动量因子的直觉

一只股票过去 60 个交易日涨了很多，可能说明：

- 基本面预期正在改善。
- 资金持续流入。
- 市场对信息反应不足。
- 趋势交易者推动趋势延续。

但也可能说明：

- 短期涨太快，接下来反转。
- 利好已被充分定价。
- 高波动股票偶然冲上去。

所以动量因子一定要用数据检验。

---

## 2. 常见动量定义

最朴素的 N 日动量：


momentum_N(t) = close(t) / close(t-N) - 1


常见窗口：

| 因子 | 含义 |
| --- | --- |
| 20 日动量 | 约 1 个月趋势 |
| 60 日动量 | 约 1 个季度趋势 |
| 120 日动量 | 约半年趋势 |

在实务中，常见改造还包括：


momentum_120_skip20 = close(t-20) / close(t-120) - 1


也就是跳过最近 20 天，避免短期反转干扰。

---

## 3. 准备 Python 环境


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(20260706)


---

## 4. 构造模拟价格数据


In [ ]:
dates = pd.bdate_range("2024-01-02", periods=280)
tickers = [f"Stock_{i:03d}" for i in range(120)]

market_ret = rng.normal(0.0003, 0.009, size=len(dates))
price = pd.DataFrame(index=dates, columns=tickers, dtype=float)

for i, ticker in enumerate(tickers):
    trend_strength = rng.normal(0.0002, 0.00025)
    beta = rng.normal(1.0, 0.25)
    noise = rng.normal(0, 0.018, size=len(dates))
    ret = trend_strength + beta * market_ret + noise
    price[ticker] = 50 * np.cumprod(1 + ret)

price.head()


---

## 5. 构造动量因子


In [ ]:
def momentum(close: pd.DataFrame, window: int) -> pd.DataFrame:
    return close / close.shift(window) - 1


mom_20 = momentum(price, 20)
mom_60 = momentum(price, 60)
mom_120 = momentum(price, 120)
mom_120_skip20 = price.shift(20) / price.shift(120) - 1

mom_20.tail()


解释：

- `mom_20` 使用过去 20 个交易日收益。
- `mom_60` 使用过去 60 个交易日收益。
- `mom_120_skip20` 使用 t-120 到 t-20 的收益，跳过最近 20 天。

---

## 6. 转成长表因子库


In [ ]:
def wide_to_long(wide: pd.DataFrame, name: str) -> pd.DataFrame:
    return (
        wide.stack(future_stack=True)
        .dropna()
        .rename(name)
        .reset_index()
        .rename(columns={"level_0": "date", "level_1": "ticker"})
    )


factor_library = wide_to_long(mom_20, "momentum_20d")
factor_library = factor_library.merge(wide_to_long(mom_60, "momentum_60d"), on=["date", "ticker"], how="outer")
factor_library = factor_library.merge(wide_to_long(mom_120, "momentum_120d"), on=["date", "ticker"], how="outer")
factor_library = factor_library.merge(wide_to_long(mom_120_skip20, "momentum_120d_skip20d"), on=["date", "ticker"], how="outer")

factor_library.head()


这就是今天的目标雏形：动量因子库。

---

## 7. 生成未来收益标签并检验 IC


In [ ]:
future_20d_ret = price.shift(-20) / price - 1
label = wide_to_long(future_20d_ret, "future_20d_ret")

data = factor_library.merge(label, on=["date", "ticker"], how="inner").dropna()

ic_table = {}
for col in ["momentum_20d", "momentum_60d", "momentum_120d", "momentum_120d_skip20d"]:
    daily_ic = data.groupby("date").apply(
        lambda g: g[col].corr(g["future_20d_ret"], method="spearman"),
        include_groups=False
    )
    ic_table[col] = {
        "mean_rank_ic": daily_ic.mean(),
        "icir": daily_ic.mean() / daily_ic.std(),
        "positive_ratio": (daily_ic > 0).mean(),
    }

ic_report = pd.DataFrame(ic_table).T
ic_report


---

## 8. 分组检查


In [ ]:
latest_date = data["date"].max()
one_day = data[data["date"] == latest_date].copy()
one_day["group"] = pd.qcut(
    one_day["momentum_60d"].rank(method="first"),
    q=5,
    labels=["G1 弱动量", "G2", "G3", "G4", "G5 强动量"]
)

group_ret = one_day.groupby("group", observed=True)["future_20d_ret"].mean()
group_ret


In [ ]:
group_ret.plot(kind="bar", title="动量分组未来 20 日平均收益")
plt.ylabel("Future 20D Return")
plt.xticks(rotation=30)
plt.show()


---

## 9. 目标产出：动量因子库函数


In [ ]:
def build_momentum_factor_library(close: pd.DataFrame) -> pd.DataFrame:
    factors = {
        "momentum_20d": close / close.shift(20) - 1,
        "momentum_60d": close / close.shift(60) - 1,
        "momentum_120d": close / close.shift(120) - 1,
        "momentum_120d_skip20d": close.shift(20) / close.shift(120) - 1,
    }

    out = None
    for name, wide in factors.items():
        part = wide_to_long(wide, name)
        out = part if out is None else out.merge(part, on=["date", "ticker"], how="outer")
    return out


momentum_library = build_momentum_factor_library(price)
momentum_library.tail()


---

## 10. 知识图谱


In [ ]:
mindmap
  root((动量因子1))
    20日动量
      近1个月趋势
    60日动量
      近1季度趋势
    120日动量
      半年趋势
    跳过窗口
      避免短期反转
    检验
      RankIC
      分组收益
      因子库


---

## 11. 作业

1. 把 60 日动量换成 40 日、80 日，观察 IC。
2. 比较 `momentum_120d` 和 `momentum_120d_skip20d`。
3. 用第 5 天的分层回测函数检验动量因子。
4. 思考：动量因子和价值因子可能在什么时候冲突？

---

## 12. 自测题

1. 20 日动量的公式是什么？  
   答案：`close(t) / close(t-20) - 1`。

2. 为什么要跳过最近 20 天？  
   答案：为了减少短期反转对中长期动量的干扰。

3. 动量因子值越大一定越好吗？  
   答案：不一定，要看 IC、分组收益和交易成本。

---

## 13. 明天预告

明天会学习 RSI 和 MACD，它们属于更典型的技术指标因子。

---

## 14. 仅供学习的提醒

本文使用模拟数据解释动量因子构建方法，不构成任何投资建议。真实研究需要考虑复权价格、停牌、涨跌停、交易成本和样本外验证。

---

# 统一高质量增强模块

> 本增强模块用于把第 10 天课程统一提升到第 1-2 天那种“能直接学习、能直接运行、能直接复盘”的密度。前面的正文保留；下面是更完整的学习版。

## A. 今日任务重新聚焦

- 主题：动量因子1
- 必做：20/60日动量
- 选做：120日动量
- 目标产出：动量因子库

今天真正要练成的不是“知道一个名词”，而是能把这个主题放进完整因子研究流水线：


原始数据
  ↓
因子构造
  ↓
预处理和对齐
  ↓
IC / ICIR / 分层回测
  ↓
形成可复用模块


你学习时可以一直问自己三句话：

1. 这个因子在经济含义上解释什么？
2. 这个因子在代码里如何被严格计算？
3. 这个因子是否真的经得起检验，而不是只在故事里成立？

## B. 一个更生动的直觉案例

过去强的股票可能继续强，也可能短期反转。动量因子就是把这种争论变成窗口收益、跳过窗口和统计检验。

这个例子背后的关键直觉是：

> 趋势不是信仰，必须被定义和检验。

因子研究不是把金融名词翻译成代码，而是把一个投资假设拆成可以被验证、被复现、被质疑的实验。

## C. 今日知识骨架


动量因子1
├── 输入数据
│   ├── 行情 / 财务 / 行业 / 市值等基础字段
│   └── 明确每个字段在当时是否可得
├── 因子定义
│   ├── 写清楚公式
│   ├── 写清楚方向
│   └── 写清楚缺失和异常值处理
├── 因子检验
│   ├── Rank IC
│   ├── ICIR
│   └── 分层回测
└── 目标产出
    └── 动量因子库


## D. 完整 Python 实验

下面这段代码是一个自包含实验。你可以单独复制到 Notebook 里运行。它的目的不是模拟真实市场，而是把今天主题的计算口径、方向、检查方法串起来。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(110)
dates = pd.bdate_range("2024-01-02", periods=220)
tickers = [f"S{i:03d}" for i in range(80)]
price = pd.DataFrame(index=dates)
for t in tickers:
    price[t] = 50 * np.cumprod(1 + rng.normal(.0003, .018, len(dates)))
factors = pd.concat({
    "mom20": price / price.shift(20) - 1,
    "mom60": price / price.shift(60) - 1,
    "mom120_skip20": price.shift(20) / price.shift(120) - 1,
}, axis=1)
print(factors.dropna().tail().iloc[:, :6].round(3))


## E. 产出验收标准

完成今天课程后，你的 `动量因子库` 至少应该满足：

1. 字段命名清晰，能看出日期、股票、因子值和标签含义。
2. 因子方向明确：值越大到底代表越好、越便宜、越强，还是越低风险。
3. 缺失值和异常值有处理口径，不把未知伪装成 0。
4. 至少有一段可重复运行的 Python 实验验证核心逻辑。
5. 能用 IC、ICIR 或分层回测中的至少一种方法做初步检查。
6. 能解释这个因子在真实研究里可能失效的原因。

如果这些检查没有过，不要急着进入下一天。因子研究里很多错误不是模型问题，而是最开始的口径、方向、对齐、缺失值处理出了问题。

## F. 常见坑深挖

### 坑 1：只记公式，不检查数据可得时点

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 2：因子方向写反，却直接进入 IC 和回测

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 3：把模拟数据里的漂亮结果当成真实市场规律

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 4：忽略缺失值、极端值和样本边界

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 5：只看单一指标，不做交叉验证

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 6：没有把目标产出封装成可复用函数

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。

## G. 强化练习

### 作业 A

用自己的话写出 `动量因子1` 的一句话定义，并标明它属于收益、风险、估值、质量、技术、流动性还是预处理模块。
### 作业 B

运行完整实验代码，记录输出结果，并解释每一列结果的金融含义。
### 作业 C

故意把因子方向取反，再重新计算结果，观察 IC 或分组表现如何变化。
### 作业 D

加入 5% 缺失值或 1% 极端值，测试你的处理逻辑是否仍然稳健。
### 作业 E

把今天的 `动量因子库` 保存成一个可以被后续课程调用的函数或表格。

## H. 面试式自测

### 问：这个主题在因子研究流水线里处于哪一步？

答：它对应 `动量因子库`，用于把原始数据转成后续 IC、ICIR、分层回测或多因子合成可以使用的中间产物。
### 问：最容易出现未来函数的地方在哪里？

答：通常出现在使用未来才披露的数据、未来价格、未来收益标签错位，或把全样本统计量用于历史截面。
### 问：为什么不能只看一个漂亮结果？

答：因为单次结果可能来自样本偶然、极端值、行业暴露、市值暴露或参数过拟合，需要多角度验证。
### 问：如何判断今天产出的模块可以进入下一步？

答：至少通过字段检查、方向检查、缺失异常检查、抽样手工验证和一个简单统计检验。

## I. 今日复盘模板


第 10 天复盘：动量因子1

1. 今天我能用一句话解释的核心概念：

2. 今天最重要的公式：

3. 代码里最容易写错的地方：

4. 我检查因子方向的方法：

5. 我检查缺失值和异常值的方法：

6. 如果把这个模块放进真实研究，我还缺什么数据：

7. 今天留下的一个问题：


## J. 和下一课的连接

下一课会继续沿着这条链路推进：前一天产出的字段或模块，会成为后一天检验、扩展或组合的输入。学习时不要把每天割裂开；真正的因子研究是一条流水线。

---

## K. 学习提醒

这一份课程仍然是教学材料，示例数据是模拟数据。真实研究需要处理真实数据源、可得时点、复权、停牌、交易成本、行业和市值暴露、样本外验证。
---

# 第 10-15 天深度加厚模块


    ## L. 为什么还要加厚这一课？

    这一课属于第 10-15 天的“工程化因子”部分：它不像 Alpha、Beta 那样只靠概念就能建立直觉，也不像 PE、ROE 那样有明确财务含义。它更依赖窗口、参数、预处理顺序和检验口径。

    所以学习 `动量因子1` 时，不能只停留在“知道公式”。你至少要完成三层理解：


    第一层：公式能写对
    第二层：参数变化后结果还能解释
    第三层：能放进统一因子流水线


    如果只学第一层，代码很快能写出来，但研究时很容易陷入“参数换一下结果就变”的困境。

    ## M. 更贴近真实研究的场景

    你发现 60 日动量在全样本有效，但 20 日动量不稳定，120 日动量在熊市里失效。此时你不能简单说“动量有效”或“动量无效”，而要回答：哪个持有期、哪个市场阶段、哪个股票池里的动量更有效。

    这类问题在真实研究中很常见：一个因子看似简单，但只要换股票池、换窗口、换持有期、换市场阶段，结果就会明显变化。成熟的研究方式不是逃避这种变化，而是把变化记录下来、解释出来。

    ## N. 参数敏感性实验

    下面这段代码是专门为本课补充的参数实验。它不追求复杂，而是训练一个习惯：

    > 不要只交一个因子结果，至少比较几组合理参数。


In [ ]:
    import numpy as np
import pandas as pd

rng = np.random.default_rng(210)
dates = pd.bdate_range("2024-01-02", periods=260)
tickers = [f"S{i:03d}" for i in range(120)]
price = pd.DataFrame(index=dates)
for t in tickers:
    ret = rng.normal(0.0003, 0.018, len(dates))
    price[t] = 50 * np.cumprod(1 + ret)

def stack(wide, name):
    return wide.stack(future_stack=True).dropna().rename(name).reset_index().rename(columns={"level_0":"date","level_1":"ticker"})

label = stack(price.shift(-20) / price - 1, "future_20d_ret")
rows = []
for w in [10, 20, 40, 60, 120]:
    fac = stack(price / price.shift(w) - 1, f"mom_{w}d")
    data = fac.merge(label, on=["date", "ticker"]).dropna()
    ic = data.groupby("date").apply(lambda g: g[f"mom_{w}d"].corr(g["future_20d_ret"], method="spearman"), include_groups=False)
    rows.append({"window": w, "mean_ic": ic.mean(), "icir": ic.mean()/ic.std(), "positive_ratio": (ic>0).mean()})
print(pd.DataFrame(rows).round(4))


    ## O. 结果该怎么写进研究笔记？

    建议你用下面这个格式记录：


    因子名称：动量因子1

    1. 使用的数据：
       - 股票池：
       - 时间区间：
       - 价格 / 财务口径：

    2. 核心参数：
       - 主参数：
       - 对照参数：

    3. 因子方向：
       - 因子值越大代表：
       - 是否需要取负号：

    4. 检验结果：
       - Rank IC：
       - ICIR：
       - 分组收益：
       - 多空表现：

    5. 稳定性：
       - 参数变化后是否稳定：
       - 分阶段是否稳定：
       - 极端行情是否失效：

    6. 结论：
       - 是否进入因子库：
       - 还需要什么后续验证：


    ## P. 额外验收清单

    `动量因子库` 如果要达到可复用标准，额外检查：

    1. 窗口参数有对照，不只展示 60 日。
2. 至少比较 20、60、120 日。
3. 说明是否跳过短期反转窗口。
4. 报告不同持有期标签下的稳定性。

    ## Q. 更深入的常见误区

    ### 误区 1：参数越多越高级

    参数多不代表研究深，很多时候只是过拟合空间更大。真正高级的是解释参数为什么合理，并证明它在相邻参数下仍然不崩。

    ### 误区 2：只看全样本平均

    全样本平均可能掩盖阶段失效。至少要分年度、分市场状态、分股票池看一次。

    ### 误区 3：把预处理当成机械步骤

    去极值、标准化、中性化会改变因子含义。每加一步，都要知道自己剥离了什么，也可能损失了什么。

    ### 误区 4：忽略交易可行性

    技术、波动率、流动性类因子往往换手更高，交易成本可能非常关键。纸面有效不等于可交易。

    ## R. 加厚作业

    1. 把本课主参数上下各调整一次，记录结果变化。
    2. 把未来收益标签从 20 日改成 5 日和 60 日，观察结论是否变。
    3. 随机删除 10% 股票样本，检查结果是否稳定。
    4. 把因子取反，确认分组结果是否镜像变化。
    5. 写一段 200 字研究结论，必须同时包含“支持证据”和“风险提示”。

    ## S. 一句话升级结论

    `动量因子1` 的高质量学习标准不是“会算”，而是：

    > 会定义、会检验、会解释参数变化，也知道它在真实交易里可能被什么击穿。
